In [1]:
import numpy as np

np.random.seed(42)

X=np.random.uniform(-2, 2, (400, 3))
y=(
    np.sin(X[:, 0]) +
    0.5*(X[:, 1]**2) -
    0.8*X[:, 2]
)
y=y.reshape(-1, 1)

In [2]:
def relu(z):
    return np.maximum(0, z)

def drelu(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def dsigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z):
    return np.tanh(z)

def dtanh(z):
    return 1 - np.tanh(z)**2

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha*z)

def dleaky_relu(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)

def softplus(z):
    return np.log(1 + np.exp(z))

def dsoftplus(z):
    return sigmoid(z)

In [3]:
def initialize(layer_dims):
    params = {}
    L = len(layer_dims) - 1

    for l in range(1, L+1):
        params["W"+str(l)] = np.random.uniform(-0.5, 0.5,
                              (layer_dims[l], layer_dims[l-1]))
        params["b"+str(l)] = np.zeros((layer_dims[l], 1))

    return params

In [5]:
def forward(X, params, activation):
    cache = {}
    A = X.T
    L = len(params)//2

    for l in range(1, L):
        Z = params["W"+str(l)] @ A + params["b"+str(l)]
        A = activation(Z)
        cache["Z"+str(l)] = Z
        cache["A"+str(l)] = A

    # Output layer (linear)
    ZL = params["W"+str(L)] @ A + params["b"+str(L)]
    cache["Z"+str(L)] = ZL

    return ZL.T, cache

In [6]:
def mse(y, y_hat):
    return np.mean((y - y_hat)**2)

In [7]:
def backward(X, y, y_hat, params, cache, activation_deriv):
    grads = {}
    L = len(params)//2
    N = X.shape[0]

    dZ = (2/N)*(y_hat - y).T

    for l in reversed(range(1, L+1)):
        A_prev = X.T if l==1 else cache["A"+str(l-1)]

        grads["dW"+str(l)] = dZ @ A_prev.T
        grads["db"+str(l)] = np.sum(dZ, axis=1, keepdims=True)

        if l > 1:
            dA_prev = params["W"+str(l)].T @ dZ
            dZ = dA_prev * activation_deriv(cache["Z"+str(l-1)])

    return grads

In [8]:
def grad_norm(matrix):
    return np.sqrt(np.sum(matrix**2))

In [9]:
def train_model(layer_dims, activation, activation_deriv, name):

    params = initialize(layer_dims)
    lr = 0.01
    epochs = 1000

    for epoch in range(epochs):

        y_hat, cache = forward(X, params, activation)
        loss = mse(y, y_hat)

        grads = backward(X, y, y_hat, params, cache, activation_deriv)

        for l in range(1, len(layer_dims)):
            params["W"+str(l)] -= lr * grads["dW"+str(l)]
            params["b"+str(l)] -= lr * grads["db"+str(l)]

        if epoch == 200:
            loss200 = loss

    first_layer_norm = grad_norm(grads["dW1"])
    last_hidden_norm = grad_norm(grads["dW"+str(len(layer_dims)-2)])

    print("\nModel:", name)
    print("Final Loss:", loss)
    print("Loss at 200:", loss200)
    print("Grad Norm L1:", first_layer_norm)
    print("Grad Norm Last Hidden:", last_hidden_norm)

    return loss, first_layer_norm, last_hidden_norm

In [10]:
train_model([3,4,1], relu, drelu, "Model A (ReLU)")
train_model([3,6,6,1], relu, drelu, "Model B (ReLU)")
train_model([3,8,8,8,8,1], relu, drelu, "Model C (ReLU)")
train_model([3,8,8,8,8,8,8,8,8,1], relu, drelu, "Model D (ReLU)")
train_model([3,8,8,8,8,8,8,8,8,1], sigmoid, dsigmoid, "Model D (Sigmoid)")


Model: Model A (ReLU)
Final Loss: 0.11154512827725767
Loss at 200: 0.49268594052687353
Grad Norm L1: 0.04521736074874138
Grad Norm Last Hidden: 0.04521736074874138

Model: Model B (ReLU)
Final Loss: 0.07288586596130892
Loss at 200: 0.3205940688052462
Grad Norm L1: 0.036608745952825275
Grad Norm Last Hidden: 0.021440804238454392

Model: Model C (ReLU)
Final Loss: 0.030451498206553077
Loss at 200: 0.8477964004861142
Grad Norm L1: 0.023876345478441547
Grad Norm Last Hidden: 0.01680066614352681

Model: Model D (ReLU)
Final Loss: 0.05318482718581298
Loss at 200: 1.6326715841070876
Grad Norm L1: 0.429783965736654
Grad Norm Last Hidden: 0.6212899111786596

Model: Model D (Sigmoid)
Final Loss: 1.7438504812896145
Loss at 200: 1.743850485090402
Grad Norm L1: 6.305806043555609e-06
Grad Norm Last Hidden: 6.3468147620036515e-06


(np.float64(1.7438504812896145),
 np.float64(6.305806043555609e-06),
 np.float64(6.3468147620036515e-06))

In [11]:
print("\nObservation Table:\n")

print("{:<10} {:<10} {:<12} {:<16} {:<22} {}".format(
    "Model", "Activation", "Final Loss", "Grad Norm L1",
    "Grad Norm Last Hidden", "Observation"))

print("-"*90)

print("{:<10} {:<10} {:<12} {:<16} {:<22} {}".format(
    "Model A", "ReLU", "0.1115", "0.045", "0.045",
    "Stable, moderate loss"))

print("{:<10} {:<10} {:<12} {:<16} {:<22} {}".format(
    "Model B", "ReLU", "0.0729", "0.036", "0.021",
    "Improved, slight grad shrink"))

print("{:<10} {:<10} {:<12} {:<16} {:<22} {}".format(
    "Model C", "ReLU", "0.0304", "0.023", "0.016",
    "Best performance, stable"))

print("{:<10} {:<10} {:<12} {:<16} {:<22} {}".format(
    "Model D", "ReLU", "0.0531", "0.4298", "0.6213",
    "Larger grads, not best"))

print("{:<10} {:<10} {:<12} {:<16} {:<22} {}".format(
    "Model D", "Sigmoid", "1.7438", "6.3e-06", "6.3e-06",
    "Vanishing gradient"))


Observation Table:

Model      Activation Final Loss   Grad Norm L1     Grad Norm Last Hidden  Observation
------------------------------------------------------------------------------------------
Model A    ReLU       0.1115       0.045            0.045                  Stable, moderate loss
Model B    ReLU       0.0729       0.036            0.021                  Improved, slight grad shrink
Model C    ReLU       0.0304       0.023            0.016                  Best performance, stable
Model D    ReLU       0.0531       0.4298           0.6213                 Larger grads, not best
Model D    Sigmoid    1.7438       6.3e-06          6.3e-06                Vanishing gradient
